## Downstream scRNA-seq analyses

#### Preparation

In [ ]:
# package dependencies and suppress unnecessary warning messages 
from numba.core.errors import NumbaDeprecationWarning, NumbaPendingDeprecationWarning, NumbaWarning
import warnings

warnings.simplefilter('ignore', category=NumbaDeprecationWarning)
warnings.simplefilter('ignore', category=NumbaPendingDeprecationWarning)
warnings.simplefilter('ignore', category=NumbaWarning)

import os
prefix_output = "./Data/results/data_integration" # for notebook 05
os.makedirs(prefix_output, exist_ok=True)

In [ ]:
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

In [ ]:
pd.set_option('display.max_columns', 40)

In [ ]:
# reproducibility
seed = 10
np.random.seed(seed)

In [ ]:
# Global settings
sc.settings.set_figure_params(
dpi=600,          # high resolution for display
dpi_save=600,     # high resolution for saved files
figsize=(4, 4),   # figure size (in inches)
frameon=False,    # removes borders
)

#### Load files and create concatenated object

In [ ]:
adata = sc.read_h5ad("./Data/results/preprocessing/normalized_and_preprocessed.h5ad")

In [ ]:
adata.shape

In [ ]:
# Converting column to type str for the ad.concat function
adata.var = adata.var.astype(str)

In [ ]:
adata.var

In [ ]:
adata.obs

In [ ]:
sample_key="geo_sample_id"

In [ ]:
adata.obs[sample_key]

### Check unintegrated data

Check if the data are normalized:

In [ ]:
# We can see that different cells have different total counts. 
adata.X.sum(axis=1)

In [ ]:
# They are also clearly not log1p normalized given the value range
adata.X.max()

Since the counts are not normalized, we first normalize them:

In [ ]:
# Normalization using CPM values and log-transformation
adata.layers["counts"] = adata.X.copy() # Copy and keep the original counts in a new layer
sc.pp.normalize_total(adata) # Normalization
sc.pp.log1p(adata) # Log-transformation
adata.layers["log1p_norm"] = adata.X.copy() # Create a new layer for the normalized counts

#### Gene variance & HVGs

In [ ]:
sc.pp.highly_variable_genes(adata, flavor='seurat')

In [ ]:
# Plot the mean and dispersion of HVGs
# The thing to check that normalisation is effective, by making sure that not all HVGs are highly expressed but they are roughly drawn from different expression levels, while having a high dispersion
plt.figure(figsize=(10, 6))
sc.pl.highly_variable_genes(adata, save=".png")

In [ ]:
# Get the most highly variable genes (HVGs)
hvgs = adata.var.index[adata.var.highly_variable]
print(f"Number of HVGs: {len(hvgs)}")
print(f"First 10 HVGs: {hvgs[:10].tolist()}")

In [ ]:
# Visualise their expression
# Notice that these genes have a much higher expression in some of the cells, but not in the majority others
sc.pl.violin(adata, hvgs[:10], jitter=0.4, alpha=0.05, multi_panel=True, save="_top10_HVGs.png")

#### Dimensionality reduction by PCA

In [ ]:
# Run PCA, we calculate 50 PC components (the default is 50)
sc.tl.pca(adata, mask_var="highly_variable", n_comps=50)

In [ ]:
# Extract a few rows and columns of the PCA matrix
print(adata.obsm["X_pca"][:10, :5])

In [ ]:
sc.pl.pca_variance_ratio(adata, n_pcs=50, log=False, show=None, save=".png")

In [ ]:
# Visualise PCA result - PC1 vs PC2, PC3 vs PC4
# Check for batch labels and QC metrics, whether any of them drive significant variation in the dataset
sc.pl.pca(
    adata,
    color=["sample"],
    dimensions=[(0, 1), (2, 3), (0, 1), (2, 3)],
    ncols=2, 
    wspace=0.4,
    hspace=0.7,
    save="_first_PCs.png",
)

# We can see that different SampleGroup showed significant variation on the PC space, butb pct_mito didn't

Calculate the neighbour graph and the umap coordinates:

In [ ]:
sc.pp.neighbors(adata, n_pcs=30)
sc.tl.umap(adata)
adata

In [ ]:
# move the unintegrated umap
adata.obsm['X_umap_non_integrated'] = adata.obsm['X_umap']

In [ ]:
sc.pl.umap(adata, color=["subgroup",sample_key], wspace=0.3, ncols=2, frameon=False, save="_non_integrated.png")

In [ ]:
sc.tl.leiden(adata, resolution=0.6, key_added="leiden")

In [ ]:
sc.pl.umap(adata, color="leiden", frameon=False, legend_loc='on data')

In [ ]:
# Duplicate the 'subgroup' column into a new 'Subtypes' column
adata.obs['Subtypes'] = adata.obs['subgroup'].copy()

# List of 'leiden' clusters that you want to label as "Normal"
normal_clusters = ['5', '11', '20', '22', '25']

# Convert 'leiden' to string (if it is not already)
adata.obs['leiden'] = adata.obs['leiden'].astype(str)

# If the 'subgroup' column is Categorical, add 'Normal' to the categories of 'Subtypes'
if adata.obs['subgroup'].dtype.name == 'category':
    # Remove 'Normal' from the categories if it already exists
    if 'Normal' in adata.obs['Subtypes'].cat.categories:
        adata.obs['Subtypes'] = adata.obs['Subtypes'].cat.remove_categories('Normal')
    
    # Add 'Normal' as a category if it does not exist
    adata.obs['Subtypes'] = adata.obs['Subtypes'].cat.add_categories('Normal')

# Rename 'subgroup_new' to "Normal" for cells in the specified clusters
adata.obs.loc[
    adata.obs['leiden'].isin(normal_clusters),
    'Subtypes'
] = 'Normal'

# Check
print(adata.obs[['leiden', 'subgroup', 'Subtypes']].head())

In [ ]:
# Remove cells belonging to the clusters labeled as "Normal"
adata_filtered = adata[~adata.obs['leiden'].astype(str).isin(['5', '11', '20', '22', '25'])].copy()

In [ ]:
# Then exclude cells with the subgroup "GP3/4"
adata_filtered = adata_filtered[adata_filtered.obs['subgroup'] != 'GP3/4'].copy()

In [ ]:
sc.pl.umap(adata_filtered, color=["subgroup",sample_key], wspace=0.3, ncols=2, frameon=False, save="_non_integrated_filtered.png")

In [ ]:
sc.pp.highly_variable_genes(adata_filtered, flavor='seurat')

In [ ]:
# Plot the mean and dispersion of HVGs
# The thing to check that normalisation is effective, by making sure that not all HVGs are highly expressed but they are roughly drawn from different expression levels, while having a high dispersion
plt.figure(figsize=(10, 6))
sc.pl.highly_variable_genes(adata_filtered, save=".png")

In [ ]:
# Run PCA, we calculate 50 PC components (the default is 50)
sc.tl.pca(adata_filtered, mask_var="highly_variable", n_comps=50)

In [ ]:
sc.pl.pca_variance_ratio(adata_filtered, n_pcs=50, log=False, show=None, save=".png")

In [ ]:
# Visualise PCA result - PC1 vs PC2, PC3 vs PC4
# Check for batch labels and QC metrics, whether any of them drive significant variation in the dataset
sc.pl.pca(
    adata_filtered,
    color=["sample"],
    dimensions=[(0, 1), (2, 3), (0, 1), (2, 3)],
    ncols=2, 
    wspace=0.4,
    hspace=0.7,
    save="_first_PCs_filtered.png",
)

# We can see that different SampleGroup showed significant variation on the PC space, butb pct_mito didn't

Plot the UMAP considering the first 10 PCs

In [ ]:
sc.pp.neighbors(adata_filtered, n_pcs=10)
sc.tl.umap(adata_filtered)
adata_filtered

In [ ]:
sc.pl.umap(adata_filtered, color=["subgroup"], wspace=0.3, ncols=2, frameon=False, save="_non_integrated_filtered.png")

In [ ]:
# Plot the expression of CALCOCO2 and OTX2 on the UMAP
sc.pl.umap(adata_filtered, color=["CALCOCO2","OTX2","subgroup"], cmap="Reds", ncols=3, size=5, frameon=False, save="_CALCOCO2_OTX2.png")

This cell identifies cells co-expressing CALCOCO2 and OTX2 and highlights them on the UMAP, while displaying all other cells in grey.

In [ ]:
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

gene1 = "CALCOCO2"
gene2 = "OTX2"

# Extract raw counts and calculate CPM
counts_gene1 = adata_filtered[:, gene1].layers["counts"].flatten()
counts_gene2 = adata_filtered[:, gene2].layers["counts"].flatten()
total_counts = adata_filtered.layers["counts"].sum(axis=1).flatten()

cpm_gene1 = counts_gene1 / total_counts * 1e6
cpm_gene2 = counts_gene2 / total_counts * 1e6

expr_gene1 = cpm_gene1 > 0
expr_gene2 = cpm_gene2 > 0

# Define binary co-expression
coexpr_mask = expr_gene1 & expr_gene2

adata_filtered.obs["coexpr_binary"] = np.where(coexpr_mask, "Co-expression", "No co-espression")

# Split the data
adata_coexpr = adata_filtered[adata_filtered.obs["coexpr_binary"] == "Co-expression"]
adata_no_coexpr = adata_filtered[adata_filtered.obs["coexpr_binary"] == "No co-espression"]

fig, ax = plt.subplots(figsize=(7,7))

# Plot No co-expression in light grey
ax.scatter(
    adata_no_coexpr.obsm["X_umap"][:,0], 
    adata_no_coexpr.obsm["X_umap"][:,1], 
    c="lightgrey", s=5, label="No co-espression"
)

# Plot Co-expression in bright red (tab:red) in the foreground
ax.scatter(
    adata_coexpr.obsm["X_umap"][:,0], 
    adata_coexpr.obsm["X_umap"][:,1], 
    c="tab:red", s=1, label="Co-expression"
)

# Custom legend with only two entries
legend_elements = [
    Line2D([0], [0], marker='o', color='w', label='No co-espression',
           markerfacecolor='lightgrey', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='Co-expression',
           markerfacecolor='tab:red', markersize=10)
]

ax.grid(False)  # Disable the grid
ax.axis('off')  # Completely remove the axes (lines, ticks, and labels)

ax.legend(handles=legend_elements, title="", fontsize=14, bbox_to_anchor=(1.05, 1))
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
ax.set_title("Co-expression CALCOCO2 and OTX2")
plt.savefig("coexpression_umap.png", dpi=600, bbox_inches='tight')  # salva in PNG
plt.show()

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statannotations.Annotator import Annotator

# Output directory
output_dir = prefix_output
os.makedirs(output_dir, exist_ok=True)

genes = ["CALCOCO2", "OTX2"]
layer_name = "log1p_norm"

# Palette
custom_palette = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple"]

# Subgroup & colors
subgroups = sorted(adata_filtered.obs["subgroup"].unique().tolist())
color_dict = dict(zip(subgroups, custom_palette[:len(subgroups)]))

for gene in genes:
    # Extract normalized expression
    expr = adata_filtered[:, gene].layers[layer_name].flatten()

    # Create DataFrame and exclude cells with expression value 0
    df = pd.DataFrame({
        "expression": expr,
        "subgroup": adata_filtered.obs["subgroup"]
    }, index=adata_filtered.obs_names)
    df = df[df["expression"] > 0].copy()

    # Save raw data as CSV
    raw_data_path = os.path.join(output_dir, f"raw_data_{gene}.csv")
    df.to_csv(raw_data_path)
    print(f"Raw data salvati in: {raw_data_path}")

    plt.figure(figsize=(5, 5))
    ax = sns.boxplot(data=df, x="subgroup", y="expression", palette=color_dict)
    ax.set_yscale("log")
    ax.set_title(f"{gene} expression")
    ax.set_ylabel("Log1p normalized expression")

    # Comparisons only vs GP3
    if "GP3" not in df["subgroup"].unique():
        print(f"GP3 non presente per il gene {gene}")
        continue
    pairs = [("GP3", g) for g in df["subgroup"].unique() if g != "GP3"]

    # Statistical annotations
    annotator = Annotator(ax, pairs, data=df, x="subgroup", y="expression")
    annotator.configure(test='Mann-Whitney', text_format='star', loc='inside', verbose=0)
    annotator.apply_and_annotate()

    plt.tight_layout()
    plt.grid(False)
    output_path = os.path.join(output_dir, f"boxplot_{gene}_log1p_GP3_vs_others.png")
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close()

    print(f"Plot saved in: {output_path}")


### CNV inference from single-cell transcriptome

This analysis visualizes the CNV profile of GP3 cells across chromosome 17, highlighting the genomic region containing CALCOCO2 and identifying potential copy-number alterations.

In [ ]:
import infercnvpy as cnv

# 1. Load and process the GTF file
gtf_file = '/archivio_immuno/NGS_master_folder/genome_refs/Homo_sapiens/refdata-gex-GRCh38-2024-A/genes/genes.gtf.gz'
gtf = pd.read_csv(gtf_file, sep='\t', comment='#', header=None)
gtf.columns = ['chromosome', 'source', 'feature', 'start', 'end', 'score', 'strand', 'frame', 'attribute']
gtf = gtf[gtf['feature'] == 'gene']  # Seleziona solo i geni
gtf['gene_name'] = gtf['attribute'].str.extract('gene_name "([^"]+)"')

# Select only the main chromosomes (1-22, X, Y)
main_chromosomes = ['chr' + str(i) for i in range(1, 23)] + ['chrX', 'chrY']
gtf_genes_sorted = gtf[gtf['chromosome'].isin(main_chromosomes)].sort_values(by=['chromosome', 'start'])

# 2. Map chromosomes to the expression data
gene_positions = gtf_genes_sorted[['gene_name', 'chromosome', 'start', 'end']]

# Add the 'chromosome' column to adata.var by mapping gene names
adata.var['chromosome'] = adata.var_names.map(
    lambda gene: gene_positions.loc[gene_positions['gene_name'] == gene, 'chromosome'].values[0]
    if gene in gene_positions['gene_name'].values else 'unknown'
)

# Add the 'start' column to adata.var by mapping gene start positions
adata.var['start'] = adata.var_names.map(
    lambda gene: gene_positions.loc[gene_positions['gene_name'] == gene, 'start'].values[0]
    if gene in gene_positions['gene_name'].values else None
)

# Add the 'end' column to adata.var by mapping gene end positions
adata.var['end'] = adata.var_names.map(
    lambda gene: gene_positions.loc[gene_positions['gene_name'] == gene, 'end'].values[0]
    if gene in gene_positions['gene_name'].values else None
)

# Verify that the information has been added correctly
print(adata.var[['chromosome', 'start', 'end']].head())

In [ ]:
# Extract gene information from the GTF
gene_info = gene_positions[gene_positions['gene_name'] == 'CALCOCO2']
gene_chr = gene_info['chromosome'].values[0]
gene_start = gene_info['start'].values[0]

# Length of chromosome 17 (GRCh38)
chromosome_17_length = 83257441

# Calculate the relative position
if gene_chr == 'chr17':
    relative_pos = gene_start / chromosome_17_length
    print(f"CALCOCO2 is located at {relative_pos:.2%} of the total length of chromosome 17.")
else:
    print(f"CALCOCO2 is not located on chromosome 17 (it is on {gene_chr})")

Focalizziamoci su GP3

In [ ]:
import infercnvpy as cnv
import scanpy as sc
import matplotlib.pyplot as plt

# 1. Subset the GP3 and Normal groups
adata_gp3 = adata[adata.obs["Subtypes"].isin(["GP3", "Normal"])].copy()

# 2. Run inferCNV on the GP3 subset in a Jupyter-safe way
cnv.tl.infercnv(
    adata_gp3,
    reference_key="Subtypes",
    reference_cat="Normal",
    lfc_clip=3,
    window_size=200,
    step=10,
    dynamic_threshold=1.5,
    exclude_chromosomes=('chrX', 'chrY'),
    chunksize=1000,   # Reduced to avoid memory crashes
    n_jobs=1,         # Use a single process in Jupyter
    inplace=True,
    key_added='cnv',  
    calculate_gene_values=True
)

# 3. Copy the results from obsm['X_cnv'] (default)
adata_gp3.obsm['cnv'] = adata_gp3.obsm['X_cnv']

# 4. Plot the CNV profile for GP3 vs Normal
cnv.pl.chromosome_heatmap(
    adata_gp3,
    groupby='Subtypes',
    vmin=-0.05,
    vmax=0.05,
    save="_CNV_GP3_vs_Normal.png"
)

In [ ]:
adata_gp3

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. Extract genes located on chromosome 17
genes_chr17 = adata_gp3.var[
    (adata_gp3.var["chromosome"] == "chr17") & (~adata_gp3.var["start"].isna())
].copy()

# 2. Sort genes by genomic position
genes_chr17_sorted = genes_chr17.sort_values(by="start")

# 3. Get indices of the sorted genes
gene_indices = adata_gp3.var_names.get_indexer(genes_chr17_sorted.index)

# 4. Extract the CNV matrix at the gene level
X_gene_cnv = adata_gp3.layers["gene_values_cnv"]

# 5. Select CNV values for the ordered genes on chromosome 17
X_chr17 = X_gene_cnv[:, gene_indices]

# 6. Calculate the mean CNV per gene (GP3 only)
mean_cnv_chr17 = np.nanmean(X_chr17, axis=0)

# 7. Get the position of CALCOCO2
if "CALCOCO2" in genes_chr17_sorted.index:
    calcococo2_start = genes_chr17_sorted.loc["CALCOCO2", "start"]
    calcococo2_end = genes_chr17_sorted.loc["CALCOCO2", "end"]
else:
    print("⚠️ CALCOCO2 non trovato su chr17!")
    calcococo2_start = calcococo2_end = None

# 8. Plot
plt.figure(figsize=(4.8, 3.5))
plt.plot(genes_chr17_sorted["start"], mean_cnv_chr17, lw=1.5, color="tab:red")
plt.xlabel("Genomic position (bp) - chr17", fontsize=12)
plt.ylabel("Log2 fold change (CNV)", fontsize=12)
#plt.title("GP3 CNV Profile - chr17", fontsize=13)
plt.grid(False)

# 9. Add lines marking the position of CALCOCO2
if calcococo2_start is not None:
    plt.axvline(x=calcococo2_start, color="grey", linestyle="--", lw=1)
    plt.text(calcococo2_start, np.nanmax(mean_cnv_chr17) + 0.003, "CALCOCO2", color="k", fontsize=12, ha="center", va="bottom")

plt.tight_layout()
plt.savefig("cnv_chr17_CALCOCO2_GP3only.png", dpi=600)
plt.show()

### Generation of pseudo-bulk profiles

After the annotation of clusters into cell identities, we often would like to perform differential expression analysis (DEA) 
between conditions within particular cell types to further characterize them. DEA can be performed at the single-cell level, 
but the obtained p-values are often inflated as each cell is treated as a sample. We know that single cells within a sample 
are not independent of each other, since they were isolated from the same environment. If we treat cells as samples, we are 
not testing the variation across a population of samples, rather the variation inside an individual one. Moreover, if a sample 
has more cells than another it might bias the results.

The current best practice to correct for this is using a pseudo-bulk approach (Squair J.W., et al 2021), which involves the 
following steps:

1) Subsetting the cell type(s) of interest to perform DEA.
2) Extracting their raw integer counts.
3) Summing their counts per gene into a single profile if they pass quality control.
4) Performing DEA if at least two biological replicates per condition are available (more replicates are recommended).

We can pseudobulk using the function decoupler.get_pseudobulk. In this example, we are interested in summing the counts but 
other modes are available, for more information check its argument mode.

In [ ]:
import decoupler as dc

In [ ]:
help(dc) # show the help function

If you encounter the following error:

`AttributeError: module 'decoupler' has no attribute 'get_pseudobulk'`

you need to downgrade `decoupler` to an earlier version. The `get_pseudobulk` function was available in previous versions of `decoupler`, but it is no longer part of the new package structure starting from `decoupler >=2.0.0`, where many functions have been moved to subpackages.

With version `1.8.0`, the following code works correctly.

In [ ]:
# Get pseudo-bulk profile
pdata = dc.get_pseudobulk(
    adata_filtered,
    sample_col=sample_key,
    groups_col='subgroup',
    layer='counts',
    mode='sum',
    min_cells=10,
    min_counts=20
)

It has generated a profile for each sample and cell type. We can plot their quality control metrics:

In [ ]:
dc.plot_psbulk_samples(pdata, groupby='subgroup', figsize=(4, 4), save="plot_psbulk_samples.png")

Now that we have generated the pseudobulk profiles for each patient and each cell type, let’s explore the variability between them. For that, we will first do some simple preprocessing and then do a PCA

In [ ]:
# Store raw counts in layers
pdata.layers['counts'] = pdata.X.copy()

# Normalize, scale and compute pca
sc.pp.normalize_total(pdata, target_sum=1e4)
sc.pp.log1p(pdata)
sc.pp.scale(pdata, max_value=10)
sc.tl.pca(pdata)

# Return raw counts to X
dc.swap_layer(pdata, 'counts', X_layer_key=None, inplace=True)

In [ ]:
sc.pl.pca(pdata, color=['subgroup'], size=300, save = "plot_psbulk_samples.png")
sc.pl.pca_variance_ratio(pdata)

When looking at the PCA, it seems like the two first components explain most of the variance and they easily separate cell types from one another. In contrast, the principle components do not seem to be associated with disease status as such.

In order to have a better overview of the association of PCs with sample metadata, let’s perform ANOVA on each PC and see whether they are significantly associated with any technical or biological annotations of our samples

In [ ]:
dc.get_metadata_associations(
    pdata,
    obs_keys = ['subgroup'],  # Metadata columns to associate to PCs
    obsm_key='X_pca',  # Where the PCs are stored
    uns_key='pca_anova',  # Where the results are stored
    inplace=True,
)

In [ ]:
dc.plot_associations(
    pdata,
    uns_key='pca_anova',  # Summary statistics from the anova tests
    obsm_key='X_pca',  # where the PCs are stored
    stat_col='p_adj',  # Which summary statistic to plot
    obs_annotation_cols = ['subgroup'], # which sample annotations to plot
    titles=['Principle component scores', 'Adjusted p-values from ANOVA'],
    figsize=(5, 5),
    n_factors=10,
    save="pca_associations.png"
)

Additionally to filtering low quality samples, we can also filter noisy expressed genes in case we want to perform downstream analyses such as DEA afterwards. Note that this step should be done at the cell type level, since each cell type may express different collection of genes.

To filter genes, we will follow the strategy implemented in the function filterByExpr from edgeR. It keeps genes that have a minimum total number of reads across samples (min_total_count), and that have a minimum number of counts in a number of samples (min_count).

We can plot how many genes do we keep, you can play with the min_count and min_total_count to check how many genes would be kept when changed:

In [ ]:
dc.plot_filter_by_expr(pdata, group='subgroup', min_count=10, min_total_count=15)

Here we can observe the frequency of genes with different total sum of counts and number of samples. The dashed lines indicate the current thresholds, meaning that only the genes in the upper-right corner are going to be kept. Filtering parameters is completely arbitrary, but a good rule of thumb is to identify bimodal distributions and split them modifying the available thresholds. In this example, with the default values we would keep a good quantity of genes while filtering potential noisy genes.

Once we are content with the threshold parameters, we can perform the actual filtering:

In [ ]:
# Obtain genes that pass the thresholds
genes = dc.filter_by_expr(pdata, group='subgroup', min_count=10, min_total_count=15)

# Filter by these genes
pdata = pdata[:, genes].copy()
pdata

### Contrast between conditions

Once we have generated robust pseudo-bulk profiles, we can compute DEA. For this example, we will perform a simple experimental 
design where we compare the gene expression between GP3 (il più aggressivo) and SHH (il meno aggressivo). We will use the python 
implementation of the framework DESeq2, but we could have used any other one (limma or edgeR for example). For a better understanding 
how it works, check DESeq2’s documentation. Note that more complex experimental designs can be used by adding more factors to the 
design_factors argument.

In [ ]:
# Import DESeq2
from pydeseq2.dds import DeseqDataSet, DefaultInference
from pydeseq2.ds import DeseqStats

#### GP3 vs. SHH

In [ ]:
# Build DESeq2 object
inference = DefaultInference(n_cpus=8)
dds = DeseqDataSet(
    adata=pdata,
    design_factors='subgroup',
    ref_level=['GP3', 'SHH'],
    refit_cooks=True,
    inference=inference,
)

In [ ]:
# Compute LFCs
dds.deseq2()

In [ ]:
# Extract contrast between COVID-19 vs normal
stat_res = DeseqStats(
    dds,
    contrast=["subgroup", 'GP3', 'SHH'],
    inference=inference,
)

In [ ]:
# Compute Wald test
stat_res.summary()

In [ ]:
# Extract results
results_df = stat_res.results_df
results_df

In [ ]:
dc.plot_volcano_df(
    results_df,
    x='log2FoldChange',
    y='padj',
    top=50,
    figsize=(10, 10),
    save="Volcano_plot_GP3_vs_SHH.png"
)

Plot the Volcano plot showing CALCOCO2.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Calculate -log10(padj)
results_df['-log10padj'] = -np.log10(results_df['padj'])

# CALCOCO2 gene
autophagy_genes = ["CALCOCO2"]

# Ensure the index is a string
results_df.index = results_df.index.astype(str)

# Filter CALCOCO2
autophagy_df = results_df[results_df.index.isin(autophagy_genes)]

# Statistically significant up-regulated genes 
autophagy_up_sig = autophagy_df[
    (autophagy_df['log2FoldChange'] > 1) &
    (autophagy_df['padj'] < 0.05)
]

# All the other genes
others_df = results_df[~results_df.index.isin(autophagy_up_sig.index)]

# Plot
plt.figure(figsize=(4.5, 4.5))

# Other genes in grey
plt.scatter(others_df['log2FoldChange'], others_df['-log10padj'], s=10,
            color='grey', alpha=0.5, label='Other genes')

# Statistically significant up-regulated genes in red
plt.scatter(autophagy_up_sig['log2FoldChange'], autophagy_up_sig['-log10padj'], s=10,
            color='red', label='Upregulated autophagy genes')

# Annotation
for gene in autophagy_up_sig.index:
    x = autophagy_up_sig.loc[gene, 'log2FoldChange']
    y = autophagy_up_sig.loc[gene, '-log10padj']
    plt.text(x+10, y, gene, fontsize=12, ha='right')

# Threshold
plt.axhline(-np.log10(0.05), color='black', linestyle='--', linewidth=1)
plt.axvline(1, color='black', linestyle='--', linewidth=1)
plt.axvline(-1, color='black', linestyle='--', linewidth=1)

plt.xlabel('log2 Fold Change')
plt.ylabel('-log10 adjusted p-value')
#plt.title('Volcano Plot – Significantly Upregulated Autophagy Genes Highlighted')
#plt.legend()
plt.grid(False)
# Save the figure in PNG format
plt.savefig('volcano_autophagy_upregulated_GP3_SHH_only_CALCOCO2.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
autophagy_df

After performing DEA, we can use the obtained gene level statistics to perform enrichment analysis. Any statistic can be used, but we recommend using the t-values instead of logFCs since t-values incorporate the significance of change in their value. We will transform the obtained t-values stored in stats to a wide matrix so that it can be used by decoupler:

In [ ]:
mat = results_df[['stat']].T.rename(index={'stat': 'GP3'})
mat

This plot visualizes the correlation between CALCOCO2 and OTX2 expression at the single-cell level, showing their relationship using a scatter plot with density estimation and a linear regression line.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import scanpy as sc
from scipy.stats import spearmanr, pearsonr

# Parameters
gene1 = "CALCOCO2"
gene2 = "OTX2"

# Pre-processing if raw data are not used
adata_source = adata_filtered.copy()

# Extract expression vectors
x = adata_source[:, gene1].X
y = adata_source[:, gene2].X

# Convert sparse matrices if necessary
x = x.toarray().flatten() if hasattr(x, "toarray") else x.flatten()
y = y.toarray().flatten() if hasattr(y, "toarray") else y.flatten()

# Filter: keep only cells in which both genes are expressed (> 0)
mask = (x > 0) & (y > 0)
x_filtered = x[mask]
y_filtered = y[mask]

# Calculate correlations
r_pearson, p_pearson = pearsonr(x_filtered, y_filtered)
r_spearman, p_spearman = spearmanr(x_filtered, y_filtered)

print(f"Pearson r = {r_pearson:.2f}, p = {p_pearson:.2e}")
print(f"Spearman r = {r_spearman:.2f}, p = {p_spearman:.2e}")

# Add jitter for visualization
x_plot = x_filtered + np.random.normal(0, 0.1, size=len(x_filtered))
y_plot = y_filtered + np.random.normal(0, 0.1, size=len(y_filtered))

# Plot
fig, ax = plt.subplots(figsize=(4, 3.1))

# Scatter with jitter
sns.scatterplot(x=x_plot, y=y_plot, alpha=0.4, s=10, edgecolor=None, ax=ax)

# Density heatmap (KDE)
sns.kdeplot(x=x_plot, y=y_plot, cmap="Reds", fill=True, thresh=0.05, alpha=0.6, ax=ax)

# Linear regression
sns.regplot(x=x_plot, y=y_plot, scatter=False, color="black", label="Regression line", line_kws={"linewidth": 1}, ax=ax)

# Display Pearson r and p-value in the upper-left corner
ax.text(0.04, 0.96, f"Pearson r = {r_pearson:.2f}\np-value = {p_pearson:.2e}",
        ha='left', va='top', transform=ax.transAxes,
        fontsize=10, bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

# Axis labels
ax.set_xlabel(f"{gene1} expression", fontsize=12)
ax.set_ylabel(f"{gene2} expression", fontsize=12)
ax.legend(loc='lower right')
ax.set_ylim(-0.7, 3.7)
ax.grid(False)

plt.tight_layout()
plt.savefig('Correlation_across_subtypes_CALCOCO2_OTX2.png', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import scanpy as sc
from scipy.stats import spearmanr, pearsonr

# Parameters
gene1 = "CALCOCO2"
gene2 = "OTX2"

# AnnData for GP3 subgroup
adata_source = adata_gp3.copy()

# Extract expression vectors
x = adata_source[:, gene1].X
y = adata_source[:, gene2].X

# Convert sparse matrices if necessary
x = x.toarray().flatten() if hasattr(x, "toarray") else x.flatten()
y = y.toarray().flatten() if hasattr(y, "toarray") else y.flatten()

# Filter: keep only cells in which both genes are expressed (> 0)
mask = (x > 0) & (y > 0)
x_filtered = x[mask]
y_filtered = y[mask]

# Calculate correlations
r_pearson, p_pearson = pearsonr(x_filtered, y_filtered)
r_spearman, p_spearman = spearmanr(x_filtered, y_filtered)

print(f"Pearson r = {r_pearson:.2f}, p = {p_pearson:.2e}")
print(f"Spearman r = {r_spearman:.2f}, p = {p_spearman:.2e}")

# Add jitter for visualization
x_plot = x_filtered + np.random.normal(0, 0.1, size=len(x_filtered))
y_plot = y_filtered + np.random.normal(0, 0.1, size=len(y_filtered))

# Plot
fig, ax = plt.subplots(figsize=(4, 3.1))

# Scatter with jitter
sns.scatterplot(x=x_plot, y=y_plot, alpha=0.4, s=10, edgecolor=None, ax=ax)

# Density heatmap (KDE)
sns.kdeplot(x=x_plot, y=y_plot, cmap="Reds", fill=True, thresh=0.05, alpha=0.6, ax=ax)

# Linear regression
sns.regplot(x=x_plot, y=y_plot, scatter=False, color="black", label="Regression line", line_kws={"linewidth": 1}, ax=ax)

# Display Pearson r and p-value in the upper-left corner
ax.text(0.04, 0.96, f"Pearson r = {r_pearson:.2f}\np-value = {p_pearson:.2e}",
        ha='left', va='top', transform=ax.transAxes,
        fontsize=10, bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

# Axis labels
ax.set_xlabel(f"{gene1} expression", fontsize=12)
ax.set_ylabel(f"{gene2} expression", fontsize=12)
ax.legend(loc='lower right')
ax.set_ylim(-0.7, 3.7)
ax.grid(False)

plt.tight_layout()
plt.savefig('Correlation_in_GP3_CALCOCO2_OTX2.png', dpi=600, bbox_inches='tight')
plt.show()